In [1]:
DATA = '../data/'
FIGS = '../results/figures/'
CACHE = '../f1_cache'

In [2]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_pickle(DATA + 'laps_clean.pkl') # load the clean lap data
print(f"loaded {len(df)} laps, {df['RaceID'].nunique()} races, {df['Circuit'].nunique()} circuits")

# Hold out the demo race entirely
demo = df[(df['Year'] == 2025) & (df['Circuit'] == 'Barcelona')]
assert len(demo) > 0, "demo filter matched nothing - check df['Circuit'].unique()"
pool = df.drop(demo.index)
print(f"demo race: {len(demo)} laps, training pool: {len(pool)} laps")

# Split BY RACE WEEKEND, never randomly by lap
races = pool['RaceID'].unique()
rng = np.random.default_rng(0)
test_races = rng.choice(races, size=int(len(races) * 0.2), replace=False) # randomly selected 20% of races for testing

train = pool[~pool['RaceID'].isin(test_races)] # the pile of laps used for training the model
test = pool[pool['RaceID'].isin(test_races)] # the pile of laps used for scoring the model's accuracy
print(f"train: {len(train)} laps / test: {len(test)} laps")

# --- Tier 1: heuristic ---
ref = (train.groupby(['Circuit', 'Driver'])['LapSeconds']
            .median().rename('pred1').reset_index())
t1 = test.merge(ref, on=['Circuit', 'Driver'], how='inner')
mae1 = mean_absolute_error(t1['LapSeconds'], t1['pred1'])

# The inner join drops test laps whose (Circuit, Driver) pair never appeared in training.
# Tier 2 must be scored on the SAME laps or the comparison is meaningless.
keep = test.set_index(['Circuit', 'Driver']).index.isin(
    ref.set_index(['Circuit', 'Driver']).index)
print(f"tier 1 scored on {keep.sum()} of {len(test)} test laps")

# --- Tier 2: linear regression ---
feat = ['TyreLife', 'LapNumber', 'AirTemp', 'TrackTemp'] # the continuous numerical values that set the movement (produces the slope)
cats = ['Compound', 'Circuit', 'Driver', 'Team'] # the categorical features which set where the lap time sits (a fixed offset for lap times)

Xtr = pd.get_dummies(train[feat + cats], columns=cats, drop_first=True) # builds the training input table
Xte = pd.get_dummies(test[feat + cats], columns=cats, drop_first=True) # builds the test input table
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0) # force the test set's column structure to match training set

lin = LinearRegression().fit(Xtr, train['LapSeconds'])
pred2 = lin.predict(Xte)
mae2 = mean_absolute_error(test['LapSeconds'], pred2) # on all test laps
mae2_fair = mean_absolute_error(test['LapSeconds'][keep], pred2[keep]) # on tier 1's laps

print(f"\nTier 1 (heuristic): {mae1:.3f} s")
print(f"Tier 2 (linear):    {mae2_fair:.3f} s  <- same laps as tier 1, this is the comparison")
print(f"improvement:        {mae1 - mae2_fair:.3f} s")
print(f"Tier 2, all laps:   {mae2:.3f} s  <- use this one against tier 3")

coef = dict(zip(Xtr.columns, lin.coef_))
print(f"\nTyreLife coefficient:  {coef['TyreLife']:+.4f} s per lap of age")
print(f"LapNumber coefficient: {coef['LapNumber']:+.4f} s per lap of race")

loaded 76394 laps, 83 races, 24 circuits
demo race: 988 laps, training pool: 75406 laps
train: 61145 laps / test: 14261 laps
tier 1 scored on 12648 of 14261 test laps

Tier 1 (heuristic): 1.595 s
Tier 2 (linear):    1.405 s  <- same laps as tier 1, this is the comparison
improvement:        0.190 s
Tier 2, all laps:   1.430 s  <- use this one against tier 3

TyreLife coefficient:  +0.0277 s per lap of age
LapNumber coefficient: -0.0485 s per lap of race


/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [5]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error

num = ['TyreLife', 'LapNumber', 'AirTemp', 'TrackTemp']
cat = ['Compound', 'Circuit', 'Driver', 'Team', 'FreshTyre']

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
enc.fit(train[cat].astype(str))

def prep(d):
    X = d[num].reset_index(drop=True).copy()
    codes = pd.DataFrame(enc.transform(d[cat].astype(str)),
                         columns=cat)
    return pd.concat([X, codes], axis=1)

# The encoder emits -1 for a category never seen in training. HistGradientBoosting
# expects non-negative codes, so check whether -1 ever actually occurs before trusting it.
print("unseen categories in test:", int((prep(test)[cat] == -1).sum().sum()))
print("unseen categories in demo:", int((prep(demo)[cat] == -1).sum().sum()))

cat_idx = list(range(len(num), len(num) + len(cat)))

model = HistGradientBoostingRegressor(
    categorical_features=cat_idx,
    early_stopping=False,
    random_state=0
).fit(prep(train), train['LapSeconds'])

mae_train = mean_absolute_error(train['LapSeconds'], model.predict(prep(train)))
mae3      = mean_absolute_error(test['LapSeconds'],  model.predict(prep(test)))
mae_demo  = mean_absolute_error(demo['LapSeconds'],  model.predict(prep(demo)))

print(f"Tier 1 (heuristic):   {mae1:.3f} s")
print(f"Tier 2 (linear):      {mae2:.3f} s")
print(f"Tier 3 (boosted):     {mae3:.3f} s")
print(f"  training error:     {mae_train:.3f} s")
print(f"  DEMO RACE:          {mae_demo:.3f} s")

print("iterations:", model.n_iter_, "of", model.max_iter)
print(model.get_params()['early_stopping'], len(model.validation_score_))

unseen categories in test: 0
unseen categories in demo: 0
Tier 1 (heuristic):   1.595 s
Tier 2 (linear):      1.430 s
Tier 3 (boosted):     1.484 s
  training error:     0.473 s
  DEMO RACE:          0.903 s
iterations: 100 of 100
False 0


In [4]:
err = abs(model.predict(prep(test)) - test['LapSeconds'].values)
u = (prep(test)[cat] == -1).any(axis=1).values
print(f"test laps: {len(test)}, laps with an unseen category: {u.sum()}")
print(f"MAE, clean laps:  {err[~u].mean():.3f} s")
print(f"MAE, unseen laps: {err[u].mean():.3f} s")
print(test.assign(err=err).groupby('RaceID')['err'].agg(['mean', 'count']).sort_values('mean'))

test laps: 14261, laps with an unseen category: 0
MAE, clean laps:  1.484 s
MAE, unseen laps: nan s
             mean  count
RaceID                  
2022_15  0.586775   1049
2023_1   0.738289    889
2025_4   0.837308    952
2024_19  0.861232    883
2024_17  0.882769    853
2022_11  0.922956    845
2024_3   0.977877    852
2022_10  1.246824    639
2024_12  1.327214    634
2023_9   1.374912   1168
2024_6   1.593166    934
2022_4   1.674030    719
2023_5   1.888259   1078
2024_8   1.954179   1155
2023_12  2.856697    666
2025_2   4.064438    945


/var/folders/sj/skx1mhmd28960x609gz911f80000gn/T/ipykernel_69965/230182457.py:5: RuntimeWarning: Mean of empty slice.
  print(f"MAE, unseen laps: {err[u].mean():.3f} s")
/Users/huytran/Library/Python/3.9/lib/python/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
